In [3]:
"""
Replication of Hubicka, Marcjasz, Weron (2019):
   'A Note on Averaging Day-Ahead Electricity Price Forecasts Across Calibration Windows'

Plus the two extensions promised in the Initial Report:
  (1) robust aggregation: median and 10% trimmed mean
  (2) inverse-MAE weighted (convex) combination over 10 windows

Requires: numpy, pandas, scipy, matplotlib
Input:    Task15_P.csv (GEFCom2014 price track, zone 1)
Output:   fig1_data.png, fig2_mae_vs_window.png, fig3_iterative_windows.png,
          fig4_dm_heatmap.png, fig5_extensions.png, results_summary.csv, dm_results.csv
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ==============================================================
# 0.  CONFIG
# ==============================================================
DEBUG = False       # True -> test on first 20 days only (fast sanity check)
CSV_PATH  = 'Dati_progetto.csv'
PRICE_COL = 'Zonal Price'
# The article uses the day-ahead forecast of the SYSTEM load -> 'Forecasted Total Load'.
# (the first notebook used 'Forecasted Zonal Load'; change here if the course wants zonal load)
LOAD_COL  = 'Forecasted Total Load'

TEST_START = pd.Timestamp('2012-12-29')
TEST_END   = pd.Timestamp('2013-12-16')   # prices of the last hours of 17.12.2013 are missing

# Windows (like the article: multiples of 7 days)
WINDOWS_26    = list(range(28, 729, 28))        # 28, 56, ..., 728  (26 windows)
WINDOWS_6     = [28, 56, 84, 714, 721, 728]     # recommended set AW(28:28:84,714:7:728)
WINDOWS_10    = [28, 56, 84, 112, 140, 700, 707, 714, 721, 728]   # 28:28:140 and 700:7:728
WINDOWS_7     = list(range(28, 729, 7))         # 28, 35, ..., 728 (101 windows, multiples of 7)
WINDOWS_ALL   = sorted(set(WINDOWS_7) | set(WINDOWS_26) | set(WINDOWS_6) | set(WINDOWS_10))

N_LIST   = [7, 28, 60]    # validation lengths for the inverse-MAE weights (60 = baseline)
DM_LAG   = 0              # 0 = standard 1-step DM variance; >0 = Bartlett/HAC correction

# ==============================================================
# 1.  LOAD AND PREPROCESS DATA
# ==============================================================
print('=' * 60)
print('STEP 1 - Loading and preprocessing')
print('=' * 60)

df = pd.read_csv(CSV_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%m%d%Y %H:%M')
df = df.sort_values('timestamp').reset_index(drop=True)
df['date'] = df['timestamp'].dt.normalize()
df['hour'] = df['timestamp'].dt.hour + 1     # 1..24, as in the article


# ---- Figure 1: prices and day-ahead load forecasts (as Fig. 1 of the article)
fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axs[0].plot(df['timestamp'], df[PRICE_COL], color='steelblue', lw=0.5)
axs[0].set_ylabel('Price ($/MWh)')
axs[1].plot(df['timestamp'], df[LOAD_COL], color='firebrick', lw=0.5)
axs[1].set_ylabel('Load forecast (MW)')
for a in axs:
    a.axvline(TEST_START, color='k', linestyle='--', lw=1)
    a.grid(alpha=0.3)
axs[0].set_title('GEFCom2014 price track: prices and day-ahead load forecasts (dashed line = start of test period)')
plt.tight_layout(); plt.savefig('fig1_data.png', dpi=150); plt.close()
print('  Saved: fig1_data.png')

prices = df.pivot_table(index='date', columns='hour', values=PRICE_COL).sort_index()
loads  = df.pivot_table(index='date', columns='hour', values=LOAD_COL).sort_index()
prices = prices.reindex(columns=range(1, 25))
loads  = loads.reindex(columns=range(1, 25))

# log needs strictly positive prices: report how many are lost
n_nonpos = int((prices <= 0).sum().sum())
print(f'  Non-positive prices (set to NaN before the log): {n_nonpos}')
prices = prices.where(prices > 0)

log_p    = np.log(prices)
log_load = np.log(loads)

# Make the calendar contiguous: a missing day must stay a NaN row, otherwise the lags
# d-1, d-2, d-7 would silently point to the wrong day.
full_idx = pd.date_range(log_p.index.min(), log_p.index.max(), freq='D')
log_p    = log_p.reindex(full_idx)
log_load = log_load.reindex(full_idx)
bad = log_p.isna().any(axis=1) | log_load.isna().any(axis=1)
last_ok = bad[~bad].index.max()                     # drop trailing incomplete days
log_p, log_load, bad = log_p.loc[:last_ok], log_load.loc[:last_ok], bad.loc[:last_ok]
log_p[bad]    = np.nan                              # incomplete days are ignored in fits
log_load[bad] = np.nan

print(f'  Days in calendar           : {len(log_p)}')
print(f'  Incomplete days (ignored)  : {int(bad.sum())}')
print(f'  Date range                 : {log_p.index[0].date()} -> {log_p.index[-1].date()}')
print(f'  Mean log-price             : {np.nanmean(log_p.values):.3f}')

# ==============================================================
# 2.  FEATURE MATRIX (precomputed once, shape: n_days x 24 x 9)
#     [const, p_{d-1}, p_{d-2}, p_{d-7}, p_min, z_t, D_Mon, D_Sat, D_Sun]
# ==============================================================
print('\n' + '=' * 60)
print('STEP 2 - Building feature matrix')
print('=' * 60)

n_days    = len(log_p)
p_vals    = log_p.values                      # (n_days, 24)
load_vals = log_load.values                   # (n_days, 24)
p_min     = p_vals.min(axis=1)                # (n_days,) min of the 24 prices of each day
dow       = log_p.index.dayofweek.values      # 0=Mon, ..., 6=Sun

N_FEAT = 9
feats  = np.full((n_days, 24, N_FEAT), np.nan)
target = p_vals.copy()

feats[:, :, 0]  = 1.0                         # intercept beta_{h,0}
feats[1:, :, 1] = p_vals[:-1]                 # p_{d-1,h}
feats[2:, :, 2] = p_vals[:-2]                 # p_{d-2,h}
feats[7:, :, 3] = p_vals[:-7]                 # p_{d-7,h}
feats[1:, :, 4] = p_min[:-1, np.newaxis]      # p_min of yesterday, same for all hours
feats[:, :, 5]  = load_vals                   # z_t = log day-ahead load forecast
# day-of-week dummies D_i, i in {1, 6, 7} = Monday, Saturday, Sunday (as in the article)
for j, k in enumerate([0, 5, 6]):
    feats[:, :, 6 + j] = (dow == k).astype(float)[:, np.newaxis]

print(f'  feats.shape  : {feats.shape}   (n_days, 24 hours, {N_FEAT} regressors incl. intercept)')
print(f'  target.shape : {target.shape}')

# ==============================================================
# 3.  ARX FIT (OLS on one hour, one window)
# ==============================================================
def forecast_day(target_idx, T):
    """OLS ARX forecasts of the log-price of day `target_idx` for all 24 hours, using the
    T days before the target day (one regression per hour, solved in one batch).
    Returns an array of 24 values (np.nan if there is not enough data)."""
    out = np.full(24, np.nan)
    start = max(target_idx - T, 7)              # first 7 days have no p_{d-7}
    if start >= target_idx:
        return out
    X = feats[start:target_idx]                 # (n, 24, 9)
    y = target[start:target_idx]                # (n, 24)
    ok = ~np.isnan(X).any(axis=(1, 2)) & ~np.isnan(y).any(axis=1)   # NaN patterns are day-level
    X, y = X[ok], y[ok]
    if len(X) < N_FEAT + 1:
        return out
    Xh = np.transpose(X, (1, 0, 2))             # (24, n, 9)
    yh = np.transpose(y, (1, 0))                # (24, n)
    beta = np.linalg.pinv(Xh) @ yh[..., np.newaxis]          # (24, 9, 1) least squares
    x_new = feats[target_idx]                   # (24, 9)
    if np.isnan(x_new).any():
        return out
    return np.einsum('hi,hi->h', x_new, beta[..., 0])

# ==============================================================
# 4.  ROLLING WINDOW LOOP (all windows needed by replication + extensions)
# ==============================================================
print('\n' + '=' * 60)
print('STEP 3 - Rolling window forecasts (this may take a few minutes)')
print('=' * 60)

test_start_idx = log_p.index.searchsorted(TEST_START)
test_end_idx   = log_p.index.searchsorted(TEST_END, side='right')
test_idx       = np.arange(test_start_idx, test_end_idx)
if DEBUG:
    test_idx = test_idx[:20]
    print(f'  DEBUG MODE: using only {len(test_idx)} test days')

print(f'  Test days: {len(test_idx)}  '
      f'({log_p.index[test_idx[0]].date()} -> {log_p.index[test_idx[-1]].date()})')
print(f'  Windows to fit: {len(WINDOWS_ALL)}  -> {WINDOWS_ALL}')
print(f'  Total ARX fits: {len(test_idx) * len(WINDOWS_ALL) * 24}')

# forecasts_all[t, w, h] = log-price forecast for test day t, window w, hour h
forecasts_all = np.full((len(test_idx), len(WINDOWS_ALL), 24), np.nan)
for t_local, t_global in enumerate(test_idx):
    if t_local % 20 == 0:
        print(f'  Day {t_local+1:>3}/{len(test_idx)}  ({log_p.index[t_global].date()})')
    for w_idx, T in enumerate(WINDOWS_ALL):
        forecasts_all[t_local, w_idx, :] = forecast_day(t_global, T)

actual_log = target[test_idx]                           # (n_test, 24)
print(f'  NaN forecasts: {int(np.isnan(forecasts_all).sum())}')

def widx(T):
    return WINDOWS_ALL.index(T)

# ==============================================================
# 5.  METRICS: MAE in log-price and in $/MWh, plus DM test
# ==============================================================
def mae_log(pred, actual):
    return np.nanmean(np.abs(pred - actual))

def mae_dollar(pred_log, actual_log):
    """MAE on raw prices ($/MWh), the scale of the MAE values in the article."""
    return np.nanmean(np.abs(np.exp(pred_log) - np.exp(actual_log)))

def daily_losses(pred_log, actual_log):
    """Multivariate DM loss: per day, sum of absolute errors (price scale) over 24 hours."""
    return np.nansum(np.abs(np.exp(pred_log) - np.exp(actual_log)), axis=1)

def dm_test(loss_A, loss_B, h_lag=DM_LAG):
    """Diebold-Mariano test on daily losses. H0: equal accuracy.
    DM > 0  <=>  A has the HIGHER loss (B is better).
    Returns (DM, two-sided p, one-sided p for H1 'A better than B')."""
    d = loss_A - loss_B
    D = len(d)
    d_bar = d.mean()
    var_d = np.var(d, ddof=1)
    for k in range(1, h_lag + 1):
        gamma_k = np.mean((d[k:] - d_bar) * (d[:-k] - d_bar))
        var_d += 2 * (1 - k / (h_lag + 1)) * gamma_k
    if var_d <= 1e-12:
        return 0.0, 1.0, 0.5
    dm = d_bar / np.sqrt(var_d / D)
    return float(dm), float(2 * (1 - stats.norm.cdf(abs(dm)))), float(stats.norm.cdf(dm))

DM_ROWS = []
def compare(A_pred, B_pred, name_A, name_B):
    dm, p2, p1 = dm_test(daily_losses(A_pred, actual_log), daily_losses(B_pred, actual_log))
    winner = name_B if dm > 0 else name_A
    sig = '***' if p2 < 0.01 else '**' if p2 < 0.05 else '*' if p2 < 0.10 else ''
    print(f'{name_A + " vs " + name_B:48s} {dm:>7.3f} {p2:>8.4f} {p1:>8.4f}  {winner} {sig}')
    DM_ROWS.append(dict(A=name_A, B=name_B, DM=dm, p_two_sided=p2, p_A_better=p1, better=winner))

# ---- 5a. Win(T) ----
mae_win_log, mae_win_dollar = {}, {}
for T in WINDOWS_ALL:
    pred = forecasts_all[:, widx(T), :]
    mae_win_log[T], mae_win_dollar[T] = mae_log(pred, actual_log), mae_dollar(pred, actual_log)

# ---- 5b/5c. Averages over window sets ----
idx_26 = [widx(T) for T in WINDOWS_26]
idx_6  = [widx(T) for T in WINDOWS_6]
idx_10 = [widx(T) for T in WINDOWS_10]
aw_26 = np.nanmean(forecasts_all[:, idx_26, :], axis=1)
aw_6  = np.nanmean(forecasts_all[:, idx_6,  :], axis=1)
aw_10 = np.nanmean(forecasts_all[:, idx_10, :], axis=1)
mae_aw_26_log, mae_aw_26_dollar = mae_log(aw_26, actual_log), mae_dollar(aw_26, actual_log)
mae_aw_6_log,  mae_aw_6_dollar  = mae_log(aw_6,  actual_log), mae_dollar(aw_6,  actual_log)

print('\n' + '=' * 60)
print('REPLICATION RESULTS')
print('=' * 60)
print(f'{"Model":30s} {"MAE (log)":>12s} {"MAE ($/MWh)":>12s}')
print('-' * 60)
print(f'{"Win(28)":30s} {mae_win_log[28]:>12.4f} {mae_win_dollar[28]:>12.4f}')
print(f'{"Win(728)":30s} {mae_win_log[728]:>12.4f} {mae_win_dollar[728]:>12.4f}')
print(f'{"AW(28:28:728)":30s} {mae_aw_26_log:>12.4f} {mae_aw_26_dollar:>12.4f}')
print(f'{"AW(28:28:84,714:7:728)":30s} {mae_aw_6_log:>12.4f} {mae_aw_6_dollar:>12.4f}')
print(f'\nBest single window: Win({min(WINDOWS_26, key=lambda T: mae_win_dollar[T])})')

imp_full = np.log(mae_win_dollar[728] / mae_aw_26_dollar) * 100
imp_6    = np.log(mae_win_dollar[728] / mae_aw_6_dollar)  * 100
print(f'AW(28:28:728) improves over Win(728) by {imp_full:.2f}% (article: about 1.2-1.35%)')
print(f'AW(28:28:84,714:7:728) improves over Win(728) by {imp_6:.2f}%')

# ==============================================================
# 6.  DM TESTS (replication)
# ==============================================================
print('\n' + '=' * 60)
print('DIEBOLD-MARIANO TESTS (loss = sum of |errors| over 24h, price scale)')
print('=' * 60)
print(f'{"A vs B":48s} {"DM":>7s} {"p(2-sided)":>8s} {"p(A<B)":>8s}')
print('-' * 90)
compare(aw_6,  forecasts_all[:, widx(728), :], 'AW_6', 'Win(728)')
compare(aw_6,  forecasts_all[:, widx(28),  :], 'AW_6', 'Win(28)')
compare(aw_26, forecasts_all[:, widx(728), :], 'AW_26', 'Win(728)')
compare(aw_6,  aw_26,                          'AW_6', 'AW_26')

# ==============================================================
# 8.  EXTENSION PART 1 - robust aggregation (median, trimmed mean)
# ==============================================================
print('\n' + '=' * 60)
print('EXTENSION 1 - Robust aggregation rules')
print('=' * 60)
median_6    = np.nanmedian(forecasts_all[:, idx_6,  :], axis=1)
median_26   = np.nanmedian(forecasts_all[:, idx_26, :], axis=1)
trimmed_26  = stats.trim_mean(forecasts_all[:, idx_26, :], 0.10, axis=1)   # 2 windows cut per tail
mae_median_6, mae_median_26 = mae_dollar(median_6, actual_log), mae_dollar(median_26, actual_log)
mae_trimmed_26 = mae_dollar(trimmed_26, actual_log)

print(f'{"Median of 6-window set":40s} {mae_median_6:>10.4f}  (AW_6  = {mae_aw_6_dollar:.4f})')
print(f'{"Median of 26-window set":40s} {mae_median_26:>10.4f}  (AW_26 = {mae_aw_26_dollar:.4f})')
print(f'{"10% trimmed mean of 26 windows":40s} {mae_trimmed_26:>10.4f}  (AW_26 = {mae_aw_26_dollar:.4f})')
print('\nDM tests, extension 1:')
compare(median_6,   aw_6,  'median_6',   'AW_6')
compare(median_26,  aw_26, 'median_26',  'AW_26')
compare(trimmed_26, aw_26, 'trimmed_26', 'AW_26')

# ==============================================================
# 9.  EXTENSION PART 2 - inverse-MAE weighted (convex) combination
#     w_i = MAE_i^-1 / sum_j MAE_j^-1 ,  w_i > 0 ,  sum_i w_i = 1
#     MAE_i = MAE of window i over the N days BEFORE the forecast day (log scale)
# ==============================================================
print('\n' + '=' * 60)
print('EXTENSION 2 - Inverse-MAE weighted combination (10 windows)')
print('=' * 60)
forecasts_10 = forecasts_all[:, idx_10, :]                 # (n_test, 10, 24)
mae_aw_10 = mae_dollar(aw_10, actual_log)
print(f'  Windows: {WINDOWS_10}')
print(f'  Arithmetic mean of the 10 windows : MAE = {mae_aw_10:.4f}')

weighted, mae_weighted = {}, {}
for N in N_LIST:
    w_pred = np.full((len(test_idx), 24), np.nan)
    for t in range(len(test_idx)):
        if t < N:                                          # not enough past days: equal weights
            w = np.ones(len(WINDOWS_10)) / len(WINDOWS_10)
        else:
            past_err = np.abs(forecasts_10[t - N:t] - actual_log[t - N:t, np.newaxis, :])
            past_mae = np.maximum(np.nanmean(past_err, axis=(0, 2)), 1e-9)      # (10,)
            inv = 1.0 / past_mae
            w = inv / inv.sum()
        w_pred[t] = np.sum(w[:, np.newaxis] * forecasts_10[t], axis=0)
    weighted[N], mae_weighted[N] = w_pred, mae_dollar(w_pred, actual_log)
    print(f'  Inverse-MAE weights, N = {N:>2d}      : MAE = {mae_weighted[N]:.4f}')

print('\nDM tests, extension 2:')
compare(aw_10, aw_6, 'AW_10_mean', 'AW_6')
for N in N_LIST:
    compare(weighted[N], aw_10, f'invMAE_N{N}', 'AW_10_mean')
    compare(weighted[N], aw_6,  f'invMAE_N{N}', 'AW_6')

# ==============================================================
# 10b. FIGURES (as in the article: Fig. 2, 3, 4) + extension bar chart
# ==============================================================
print('\n' + '=' * 60)
print('STEP 4 - Figures')
print('=' * 60)

# ---- Figure 2: MAE vs window length (ARX) ----
aw_all7 = np.nanmean(forecasts_all[:, [widx(T) for T in WINDOWS_7], :], axis=1)
mae_aw_all7 = mae_dollar(aw_all7, actual_log)
cum_short = [mae_dollar(np.nanmean(forecasts_all[:, [widx(x) for x in WINDOWS_7 if x <= T], :], axis=1), actual_log)
             for T in WINDOWS_7]          # AW(28:7:T)
cum_long  = [mae_dollar(np.nanmean(forecasts_all[:, [widx(x) for x in WINDOWS_7 if x >= T], :], axis=1), actual_log)
             for T in WINDOWS_7]          # AW(T:7:728)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(WINDOWS_7, [mae_win_dollar[T] for T in WINDOWS_7], 'o', color='steelblue', ms=4, label='Win(T)')
ax.plot(WINDOWS_7, cum_short, '-', color='darkorange', lw=1.3, label='AW(28:7:T)')
ax.plot(WINDOWS_7, cum_long,  '-', color='purple', lw=1.3, label='AW(T:7:728)')
ax.axhline(mae_aw_all7,      color='firebrick', linestyle='--', label='AW(28:7:728) all 101 windows')
ax.axhline(mae_aw_26_dollar, color='gray', linestyle=':', label='AW(28:28:728)')
ax.axhline(mae_aw_6_dollar,  color='forestgreen', linestyle='--', label='AW(28:28:84,714:7:728)')
ax.set_xlabel('Calibration window T (days)'); ax.set_ylabel('MAE ($/MWh)')
ax.set_title('MAE vs calibration window length (ARX, GEFCom2014 zone 1)')
ax.legend(loc='upper right', fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('fig2_mae_vs_window.png', dpi=150); plt.close()
print('  Saved: fig2_mae_vs_window.png')

# ---- Figure 3: iterative procedure, T = {28:m:*, *:n:728}, m,n in {7, 28} ----
def iterative_sets(m, n):
    """Start from {28, 728}; add two windows at a time: one m days longer than the longest
    short window and one n days shorter than the shortest long window."""
    sets, cur, s_, l_ = [], {28, 728}, 28, 728
    sets.append(sorted(cur))
    while True:
        s_ += m; l_ -= n
        if s_ > l_:
            break
        cur = cur | {s_, l_}
        sets.append(sorted(cur))
        if s_ == l_:
            break
    return sets

fig, ax = plt.subplots(figsize=(9, 5))
styles = {(28, 7): ('o', 'steelblue'), (28, 28): ('x', 'firebrick'),
          (7, 7): ('s', 'darkorange'), (7, 28): ('^', 'purple')}
fig3_rows = []
for (m, n), (mk, col) in styles.items():
    xs, ys = [], []
    for st in iterative_sets(m, n):
        pred = np.nanmean(forecasts_all[:, [widx(T) for T in st], :], axis=1)
        xs.append(len(st)); ys.append(mae_dollar(pred, actual_log))
        fig3_rows.append(dict(m=m, n=n, n_windows=len(st), MAE_dollar=ys[-1]))
    ax.plot(xs, ys, marker=mk, color=col, ms=4, lw=0.8, mfc='none' if mk == 'o' else col,
            label=f'T={{28:{m}:*, *:{n}:728}}')
ax.axhline(mae_win_dollar[728], color='k', linestyle='--', lw=1, label='Win(728)')
ax.set_xscale('log', base=2)
ax.set_xlabel('Number of windows in T'); ax.set_ylabel('MAE ($/MWh)')
ax.set_title('MAE vs number of averaged windows (ARX)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('fig3_iterative_windows.png', dpi=150); plt.close()
pd.DataFrame(fig3_rows).to_csv('fig3_values.csv', index=False)
best3 = min(fig3_rows, key=lambda r: r['MAE_dollar'])
print(f'  Saved: fig3_iterative_windows.png  (lowest MAE: m={best3["m"]}, n={best3["n"]}, '
      f'{best3["n_windows"]} windows, {best3["MAE_dollar"]:.4f})')

# ---- Figure 4: pairwise DM heatmap ----
heat_sets = {
    'Win(28)': forecasts_all[:, widx(28), :],
    'Win(364)': forecasts_all[:, widx(364), :],
    'Win(728)': forecasts_all[:, widx(728), :],
    'AW(364,728)': np.nanmean(forecasts_all[:, [widx(364), widx(728)], :], axis=1),
    'AW(28,728)': np.nanmean(forecasts_all[:, [widx(28), widx(728)], :], axis=1),
    'AW(28,56,728)': np.nanmean(forecasts_all[:, [widx(28), widx(56), widx(728)], :], axis=1),
    'AW(28:28:728)': aw_26,
    'AW(28:7:728)': aw_all7,
    'AW(28:28:84,714:7:728)': aw_6,
    'Median_6': median_6,
    'Trimmed_26': trimmed_26,
    'AW_10_mean': aw_10,
    'invMAE_N60': weighted[60],
}
names = sorted(heat_sets, key=lambda k: -mae_dollar(heat_sets[k], actual_log))   # worst first
losses = {k: daily_losses(heat_sets[k], actual_log) for k in names}
k_ = len(names)
img = np.zeros((k_, k_, 3))          # black = not significant
for i, row in enumerate(names):      # row = worse set (Y axis)
    for j, col in enumerate(names):  # col = better set (X axis)
        if i == j:
            continue
        p = dm_test(losses[col], losses[row])[2]       # one-sided: col better than row
        if p < 0.05:                                    # green (dark) -> yellow
            f = p / 0.05
            img[i, j] = (0.9 * f, 0.5 + 0.4 * f, 0.1 * (1 - f))
        elif p < 0.10:                                  # weak significance: red
            img[i, j] = (0.85, 0.1, 0.1)
fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(img, interpolation='nearest')
for i in range(k_):
    ax.text(i, i, 'x', ha='center', va='center', color='white', fontsize=9)
ax.set_xticks(range(k_)); ax.set_xticklabels(names, rotation=90, fontsize=8)
ax.set_yticks(range(k_)); ax.set_yticklabels(names, fontsize=8)
ax.set_xlabel('Better set (X axis)'); ax.set_ylabel('Worse set (Y axis)')
ax.set_title('Multivariate DM test, one-sided p-values (green/yellow: p<5%, red: 5-10%, black: n.s.)', fontsize=9)
plt.tight_layout(); plt.savefig('fig4_dm_heatmap.png', dpi=150); plt.close()
print('  Saved: fig4_dm_heatmap.png')

# ---- Figure 5: extensions (bar chart of MAE) ----
bars = [('AW_6', mae_aw_6_dollar), ('Median_6', mae_median_6),
        ('AW_26', mae_aw_26_dollar), ('Median_26', mae_median_26), ('Trimmed_26', mae_trimmed_26),
        ('AW_10', mae_aw_10)] + [(f'invMAE N={N}', mae_weighted[N]) for N in N_LIST]
fig, ax = plt.subplots(figsize=(9, 5))
vals = [v for _, v in bars]
ax.bar([b for b, _ in bars], vals, color=['forestgreen', 'seagreen', 'firebrick', 'indianred', 'salmon',
                                         'steelblue'] + ['darkorange'] * len(N_LIST))
ax.set_ylim(min(vals) * 0.98, max(vals) * 1.01)
ax.set_ylabel('MAE ($/MWh)'); ax.set_title('Extensions: aggregation rules vs arithmetic mean (ARX)')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('fig5_extensions.png', dpi=150); plt.close()
print('  Saved: fig5_extensions.png')

# ==============================================================
# 10. SAVE SUMMARY
# ==============================================================
print('\n' + '=' * 60)
print('STEP 5 - Saving summary')
print('=' * 60)
rows = [('Win(28)', mae_win_log[28], mae_win_dollar[28]),
        ('Win(728)', mae_win_log[728], mae_win_dollar[728]),
        ('AW(28:28:728)', mae_aw_26_log, mae_aw_26_dollar),
        ('AW(28:28:84,714:7:728)', mae_aw_6_log, mae_aw_6_dollar),
        ('Median_6', mae_log(median_6, actual_log), mae_median_6),
        ('Median_26', mae_log(median_26, actual_log), mae_median_26),
        ('Trimmed_26', mae_log(trimmed_26, actual_log), mae_trimmed_26),
        ('AW_10_mean', mae_log(aw_10, actual_log), mae_aw_10)]
rows += [(f'InverseMAE_N{N}', mae_log(weighted[N], actual_log), mae_weighted[N]) for N in N_LIST]
pd.DataFrame(rows, columns=['Model', 'MAE_log', 'MAE_dollar']).to_csv('results_summary.csv', index=False)
pd.DataFrame(DM_ROWS).to_csv('dm_results.csv', index=False)
print('  Saved: results_summary.csv, dm_results.csv')
print('\nDone.')

STEP 1 - Loading and preprocessing
  Saved: fig1_data.png
  Non-positive prices (set to NaN before the log): 0
  Days in calendar           : 1081
  Incomplete days (ignored)  : 1
  Date range                 : 2011-01-01 -> 2013-12-16
  Mean log-price             : 3.777

STEP 2 - Building feature matrix
  feats.shape  : (1081, 24, 9)   (n_days, 24 hours, 9 regressors incl. intercept)
  target.shape : (1081, 24)

STEP 3 - Rolling window forecasts (this may take a few minutes)
  Test days: 353  (2012-12-29 -> 2013-12-16)
  Windows to fit: 101  -> [28, 35, 42, 49, 56, 63, 70, 77, 84, 91, 98, 105, 112, 119, 126, 133, 140, 147, 154, 161, 168, 175, 182, 189, 196, 203, 210, 217, 224, 231, 238, 245, 252, 259, 266, 273, 280, 287, 294, 301, 308, 315, 322, 329, 336, 343, 350, 357, 364, 371, 378, 385, 392, 399, 406, 413, 420, 427, 434, 441, 448, 455, 462, 469, 476, 483, 490, 497, 504, 511, 518, 525, 532, 539, 546, 553, 560, 567, 574, 581, 588, 595, 602, 609, 616, 623, 630, 637, 644, 651, 658, 66